In [1]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning


ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)


from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.codi.models import CODI


warnings.filterwarnings("ignore", category=ConvergenceWarning)          # sklearn MLP
warnings.filterwarnings("ignore", message="Parameters: {")              # XGBoost unused params
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")


dataset_name = "nursery"   

dataset_path = ROOT / "raw_data" / f"{dataset_name}.csv"
output_path = ROOT / "discretized_data" / f"{dataset_name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess (same style you used for nursery)
print(f"Discretizing {dataset_path} -> {output_path}")
discretize_preprocess(str(dataset_path), str(output_path))

# Paths for pipeline
input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "codi")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

class CoDiCar(CODI):
    def __init__(self):
        super().__init__(
            # Diffusion hyperparameters
            n_steps=50,        # you can increase (e.g. 100) if you want stronger sampling
            beta_1=1e-5,
            beta_T=0.02,

            # Network architecture
            encoder_dim_con=(64, 128, 256),
            encoder_dim_dis=(64, 128, 256),
            nf_con=16,
            nf_dis=64,
            activation="relu",

            # Training hyperparameters
            epochs=30,         # bump up (e.g. 50) if you want more training for car (1781 rows)
            batch_size=512,
            lr_con=2e-3,
            lr_dis=2e-3,
            grad_clip=1.0,

            # Contrastive learning weights
            lambda_con=0.2,
            lambda_dis=0.2,

            # Misc
            random_state=42,
            device=None,       # auto: cuda if available, otherwise cpu
        )

pipeline = TrainTestSplitPipeline(
    model=lambda: CoDiCar()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print("\nPipeline result (should include TSTR metrics + result path):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\nursery.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\nursery.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\nursery
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\nursery
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\codi
Loaded data with shape: (12960, 9)


INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Training CoDi Model
INFO:katabatic.models.codi.models:================================================================================


Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)


INFO:katabatic.models.codi.models:Loaded training data: (10368, 9)
INFO:katabatic.models.codi.models:Schema: 0 continuous, 9 categorical columns
INFO:katabatic.models.codi.models:Building models: con_dim=1, cat_dim=32
INFO:katabatic.models.codi.models:Continuous model params: 166,434
INFO:katabatic.models.codi.models:Discrete model params: 359,168
INFO:katabatic.models.codi.models:
Training for 30 epochs...
INFO:katabatic.models.codi.models:Epoch 1/30: loss_con=0.0000, loss_dis=62.0669
INFO:katabatic.models.codi.models:Epoch 5/30: loss_con=0.0000, loss_dis=61.1028
INFO:katabatic.models.codi.models:Epoch 10/30: loss_con=0.0000, loss_dis=60.8220
INFO:katabatic.models.codi.models:Epoch 15/30: loss_con=0.0000, loss_dis=60.6811
INFO:katabatic.models.codi.models:Epoch 20/30: loss_con=0.0000, loss_dis=60.6015
INFO:katabatic.models.codi.models:Epoch 25/30: loss_con=0.0000, loss_dis=60.5733
INFO:katabatic.models.codi.models:Epoch 30/30: loss_con=0.0000, loss_dis=60.5250
INFO:katabatic.models.co


Results saved to: Results\nursery\codi_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.4703
F1 Score: 0.4583

MLP:
Accuracy: 0.6902
F1 Score: 0.6847

RF:
Accuracy: 0.7191
F1 Score: 0.7115

XGBoost:
Accuracy: 0.7029
F1 Score: 0.6966

Pipeline result (should include TSTR metrics + result path):
Train test split pipeline executed successfully.
